# Kumo Engine task examples with AdventureWorks in Google Colab

Explore binary classification, regression, and multiclass classification with Kumo Engine using AdventureWorks data. The notebook trains and scores a 90-day repeat-order binary classification task by default; regression and multiclass queries are commented alternatives. It follows the **prepare → connect → define tables/graph → query → train/reload → predict** workflow and uses the Linux `FileConnector` with `Customer`, `Product`, `SalesOrderHeader`, and `SalesOrderDetail`.

Download the original `Customer.csv`, `Product.csv`, `SalesOrderHeader.csv`, and `SalesOrderDetail.csv` exports yourself from [Microsoft's AdventureWorks OLTP sample files](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script), then place them in `SOURCE_DATA_DIR` inside Colab or mounted Drive. The notebook does not download or bundle the source data.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.

Set the data paths below to locations available inside your Colab runtime. If the files are in Google Drive, mount Drive in Colab before running the relevant cells. The preparation step creates separate, labeled CSVs and does not modify the source files.

**Runtime target:** Linux x86-64 Google Colab runtime version 2026.07 (Python 3.12.13) with a CUDA-capable GPU. GPU model, RAM, and local disk vary by Colab allocation; verify them in the runtime. Install Kumo Engine from PyPI once, restart when instructed, then resume at Configuration.

**Compute and storage:** provide enough GPU memory and CPU RAM for the graph and model. Plan for at least four times the materialized graph-index size in free local disk, plus room for both raw and prepared copies of the four files, the model bundle and predictions. This is planning guidance, not a measured peak. `BASE` and Kumo scratch use `/content` local disk; it is temporary. Copy required retained artifacts to Drive or download them before the runtime ends.

**Permissions and access:** allow PyPI package installation, read the four user-supplied source files, and write under `/content`. If `SOURCE_DATA_DIR` is on Google Drive, mount and authorize Drive before running file checks; Drive access is optional and is not a substitute for local scratch. See [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html) for the current 2026.07 image details.


## Install Kumo Engine from PyPI (once per new Colab runtime)

In [ ]:
import platform
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Choose a Python 3.12 Colab runtime; found {sys.version.split()[0]}.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Select a Linux x86-64 Colab runtime.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "kumo-engine"])
print("Restart the Colab runtime once, then resume at Configuration.")


## Configuration

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

MODE = "train"  # "train" or "reload" from a saved local bundle.
MODEL_BUNDLE_PATH = None  # For reload, set to Path("<path/to/model_bundle>").
SOURCE_DATA_DIR = Path("<path/to/adventureworks_csvs>")
BASE = Path("/content/kumo-workspace")  # Local runtime disk; not mounted Drive.
RAW_DATA_DIR = BASE / "raw"
PREPARED_DATA_DIR = BASE / "prepared"
SCRATCH_DIR = BASE / "scratch"
ARTIFACT_ROOT = BASE / "artifacts"
PREDICTION_TIME = None  # Derived from the supplied order dates during preparation.

SAVE_PREDICTIONS = True

if MODE not in {"train", "reload"}:
    raise ValueError("MODE must be 'train' or 'reload'.")
runtime_root = Path("/content").resolve()
for name, path in {"BASE": BASE, "SCRATCH_DIR": SCRATCH_DIR}.items():
    resolved = path.expanduser().resolve()
    if not resolved.is_relative_to(runtime_root):
        raise ValueError(f"{name} must be under /content.")
    existing_parent = resolved
    while not existing_parent.exists():
        existing_parent = existing_parent.parent
    if existing_parent.stat().st_dev != runtime_root.stat().st_dev:
        raise ValueError(f"{name} must use local Colab disk, not mounted Drive.")
for name, path in {"SOURCE_DATA_DIR": SOURCE_DATA_DIR, "BASE": BASE}.items():
    if "<" in str(path):
        raise ValueError(f"Set {name} to a path available inside the Colab runtime.")


## Initialize the local Kumo runtime

In [ ]:
# Create local scratch and retained-artifact directories, test write access, configure Kumo, and verify GPU access.
import csv
import json
import shutil
import tempfile
from importlib.metadata import version
from uuid import uuid4

import pandas as pd
import torch
from IPython.display import display

import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumoapi.graph import Edge

scratch_root = SCRATCH_DIR.expanduser().resolve()
scratch_root.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=str(scratch_root))
import sys
print("Python:", sys.version.split()[0])
print("Kumo:", kumo.__version__, "| GA package:", version("kumo-engine"))
print("Scratch:", scratch_root)
print("Free local disk GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))
if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU-backed Colab runtime before materializing the graph.")
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    print("GPU:", device.name, "VRAM GiB:", round(device.total_memory / 2**30, 1))

def disk_checkpoint(label):
    free_gib = round(shutil.disk_usage(scratch_root).free / 2**30, 2)
    print(f"Disk after {label}: {free_gib} GiB free")
    if free_gib < 5:
        raise RuntimeError("Less than 5 GiB remains in local Colab storage; stop before the next step.")


## Verify access to the four AdventureWorks files

SHA-256 checks are optional for separately obtained exports. The notebook still checks that the source and local copy have identical bytes; later cells validate the expected table shape.


In [ ]:
import hashlib
import shutil

# Optional: set a SHA-256 value for any file whose exact bytes must be pinned.
# Leave as None when using a separately obtained AdventureWorks export.
EXPECTED_DATA_SHA256 = {
    "Customer.csv": None,
    "Product.csv": None,
    "SalesOrderHeader.csv": None,
    "SalesOrderDetail.csv": None,
}

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not SOURCE_DATA_DIR.is_dir():
    raise FileNotFoundError(f"Source directory is not visible: {SOURCE_DATA_DIR}")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, expected_hash in EXPECTED_DATA_SHA256.items():
    source = SOURCE_DATA_DIR / name
    if not source.is_file():
        raise FileNotFoundError(f"Missing AdventureWorks source file: {source}")
    source_hash = sha256(source)
    if expected_hash is not None and source_hash != expected_hash:
        raise RuntimeError(f"{name} does not match its configured SHA-256 checksum.")
    destination = RAW_DATA_DIR / name
    if destination.exists():
        if sha256(destination) != source_hash:
            raise RuntimeError(f"Local file differs from source at {destination}; use a clean workspace.")
    elif source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
        if sha256(destination) != source_hash:
            raise RuntimeError(f"Copied file differs from source at {destination}.")
    print("Verified AdventureWorks file:", destination)


## Prepare the four headerless TSV exports as local CSV tables

The four source files stay unchanged. The prepared directory contains only selected columns from each file; no customer ID is added to `SalesOrderDetail` and no new graph relationship is created. Each input row is checked for its expected field count.

In [ ]:
# (expected raw field count, selected column name -> zero-based raw field index)
TABLE_LAYOUTS = {
    'Customer': (7, {
        'CustomerID': 0, 'StoreID': 2, 'TerritoryID': 3,
    }),
    'Product': (25, {
        'ProductID': 0, 'Name': 1, 'Color': 5, 'ProductLine': 15,
        'Class': 16, 'StandardCost': 8, 'ListPrice': 9,
    }),
    'SalesOrderHeader': (26, {
        'SalesOrderID': 0, 'CustomerID': 10, 'OrderDate': 2,
        'OnlineOrderFlag': 6, 'TerritoryID': 12,
        'SubTotal': 19, 'TotalDue': 22,
    }),
    'SalesOrderDetail': (11, {
        'SalesOrderDetailID': 1, 'SalesOrderID': 0, 'ProductID': 4,
        'ModifiedDate': 10, 'OrderQty': 3, 'UnitPrice': 6,
        'LineTotal': 8,
    }),
}

PREPARED_DATA_DIR.mkdir(parents=True, exist_ok=True)
row_counts = {}
date_bounds = [None, None]
for table_name, (expected_fields, columns) in TABLE_LAYOUTS.items():
    raw_path = RAW_DATA_DIR / f'{table_name}.csv'
    prepared_path = PREPARED_DATA_DIR / f'{table_name}.csv'
    if not raw_path.is_file():
        raise FileNotFoundError(f'Missing uploaded table: {raw_path}')
    temporary_path = prepared_path.with_suffix('.csv.tmp')
    row_count = 0
    with raw_path.open(newline='', encoding='utf-8-sig') as raw_file, \
         temporary_path.open('w', newline='', encoding='utf-8') as prepared_file:
        reader = csv.reader(raw_file, delimiter='\t')
        writer = csv.writer(prepared_file)
        writer.writerow(columns)
        for row_count, row in enumerate(reader, start=1):
            if len(row) != expected_fields:
                raise ValueError(f'{raw_path.name} row {row_count}: expected '
                                 f'{expected_fields} fields, found {len(row)}')
            selected = [row[index] for index in columns.values()]
            if table_name == 'SalesOrderHeader':
                order_date = row[2][:10]
                date_bounds[0] = min(date_bounds[0], order_date) if date_bounds[0] else order_date
                date_bounds[1] = max(date_bounds[1], order_date) if date_bounds[1] else order_date
            for date_column in ('OrderDate', 'ModifiedDate'):
                if date_column in columns:
                    selected[list(columns).index(date_column)] = row[columns[date_column]][:10]
            writer.writerow(selected)
    if row_count == 0:
        raise ValueError(f'{raw_path.name} is empty')
    temporary_path.replace(prepared_path)
    row_counts[table_name] = row_count
print('Prepared rows:', row_counts)
print('OrderDate range:', *date_bounds)
PREDICTION_TIME = datetime.fromisoformat(date_bounds[1]) + timedelta(days=1)
print('Scoring boundary (day after latest OrderDate):', PREDICTION_TIME)
connector = kumo.FileConnector(root_dir=str(PREPARED_DATA_DIR))
print('Local tables:', connector.table_names())

## Connect to the local files

In [ ]:
connector = kumo.FileConnector(root_dir=str(PREPARED_DATA_DIR))
print("Local tables:", connector.table_names())
display(pd.DataFrame([
    {"column": column.name, "physical_type": column.dtype, "semantic_type": column.stype}
    for column in connector["SalesOrderHeader"].columns
]))


## Define the four AdventureWorks tables

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector['Customer'],
        ['CustomerID', 'StoreID', 'TerritoryID'],
        primary_key='CustomerID',
    )
    .with_column('StoreID', stype=Stype.categorical)
    .with_column('TerritoryID', stype=Stype.categorical)
)

product = (
    kumo.Table.from_source_table(
        connector['Product'],
        ['ProductID', 'Name', 'Color', 'ProductLine', 'Class',
         'StandardCost', 'ListPrice'],
        primary_key='ProductID',
    )
    .with_column('Name', stype=Stype.text)
    .with_column('Color', stype=Stype.categorical)
    .with_column('ProductLine', stype=Stype.categorical)
    .with_column('Class', stype=Stype.categorical)
    .with_column('StandardCost', stype=Stype.numerical)
    .with_column('ListPrice', stype=Stype.numerical)
)

sales_order_header = (
    kumo.Table.from_source_table(
        connector['SalesOrderHeader'],
        ['SalesOrderID', 'CustomerID', 'OrderDate', 'OnlineOrderFlag',
         'TerritoryID', 'SubTotal', 'TotalDue'],
        primary_key='SalesOrderID', time_column='OrderDate',
    )
    .with_column('CustomerID', stype=Stype.ID)
    .with_column('OrderDate', dtype=Dtype.date)
    .with_column('OnlineOrderFlag', stype=Stype.categorical)
    .with_column('TerritoryID', stype=Stype.categorical)
    .with_column('SubTotal', stype=Stype.numerical)
    .with_column('TotalDue', stype=Stype.numerical)
)

`SalesOrderDetail` uses its original `ModifiedDate` as the line-item time column. The prepared file retains only columns selected from the source; verify the timestamp's meaning before treating model metrics as a benchmark.

In [ ]:
sales_order_detail = (
    kumo.Table.from_source_table(
        connector['SalesOrderDetail'],
        ['SalesOrderDetailID', 'SalesOrderID', 'ProductID', 'ModifiedDate',
         'OrderQty', 'UnitPrice', 'LineTotal'],
        primary_key='SalesOrderDetailID', time_column='ModifiedDate',
    )
    .with_column('SalesOrderID', stype=Stype.ID)
    .with_column('ProductID', stype=Stype.ID)
    .with_column('ModifiedDate', dtype=Dtype.date)
    .with_column('OrderQty', stype=Stype.numerical)
    .with_column('UnitPrice', stype=Stype.numerical)
    .with_column('LineTotal', stype=Stype.numerical)
)

## Validate the graph and materialize a snapshot

In [ ]:
graph = kumo.Graph(
    tables=[customer, product, sales_order_header, sales_order_detail],
    edges=[
        Edge(src_table='SalesOrderHeader', fkey='CustomerID', dst_table='Customer'),
        Edge(src_table='SalesOrderDetail', fkey='SalesOrderID', dst_table='SalesOrderHeader'),
        Edge(src_table='SalesOrderDetail', fkey='ProductID', dst_table='Product'),
    ],
)
graph.validate()
print('Graph ID:', graph.id)
print('Tables:', list(graph._named_tables))
print('Edges:', len(graph.edges))
disk_checkpoint('graph validation')

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)
disk_checkpoint("snapshot")

## Define the predictive query

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context.

The active query predicts whether a customer who ordered in the past year will order again in the next 90 days. It uses the existing `SalesOrderHeader.CustomerID → Customer.CustomerID` relationship and does not modify `SalesOrderDetail`.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        'PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) > 0 '
        'FOR EACH Customer.CustomerID '
        'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
    ),
)
pquery.validate()
print('Task type:', pquery.get_task_type())
print('Prediction time:', PREDICTION_TIME)

### Optional task alternatives

The next two query cells are commented-out alternatives. To switch tasks, comment out the active query above and uncomment exactly one alternative. Keep `MODE = "train"` when changing tasks because an existing model bundle belongs to its original query.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         'PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) '
#         'FOR EACH Customer.CustomerID '
#         'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
#     ),
# )
# pquery.validate()
# print('Task type:', pquery.get_task_type())

In [ ]:
# Multiclass classification: predict the territory of an active customer's next order.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         'PREDICT FIRST(SalesOrderHeader.TerritoryID, 0, 90, days) '
#         'FOR EACH Customer.CustomerID '
#         'WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0'
#     ),
# )
# pquery.validate()
# print('Task type:', pquery.get_task_type())


### Generate a Training Table

After validating the forecast query, bind the training table to a snapshot of the current graph. The next cell refreshes `snapshot` if it belongs to an older graph schema; the reload route skips training-table generation.

In [ ]:
train_table = None
if MODE == "train":
    current_snapshot = globals().get("snapshot")
    snapshot_graph_id = getattr(current_snapshot, "id", "").split("@", 1)[0]
    if snapshot_graph_id != graph.id:
        print("Refreshing stale or missing graph snapshot...")
        snapshot = graph.snapshot()
    assert snapshot.id.split("@", 1)[0] == graph.id
    print("Graph:", graph.id)
    print("Snapshot:", snapshot.id)
    train_table = pquery.generate_training_table(snapshot)
    print("Training table:", train_table.train_table_path)
    disk_checkpoint("training-table generation")
else:
    print("Reload route: training-table generation skipped.")


## Create a Trainer

For `MODE = "train"`, the suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting. For `MODE = "reload"`, load the retained model bundle instead.

In [ ]:
if MODE == "train":
    model_plan = pquery.suggest_model_plan(snapshot)
    model_plan.optimization.max_epochs = 5
    # Create a Trainer with your model plan.
    trainer = kumo.Trainer(model_plan)
    print(model_plan)
    print("Validation selection metric:", getattr(
        model_plan.training_job, "tune_metric", "not exposed by this build"))
else:
    if not MODEL_BUNDLE_PATH or not Path(MODEL_BUNDLE_PATH).is_dir():
        raise ValueError("Set MODEL_BUNDLE_PATH to a retained local model bundle directory.")
    trainer = kumo.Trainer.load(Path(MODEL_BUNDLE_PATH))
    print("Loaded:", MODEL_BUNDLE_PATH)


### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. Then retain the complete model bundle outside the temporary working directory.

In [ ]:
training_result = None
retained_bundle = Path(MODEL_BUNDLE_PATH) if MODE == "reload" else None
if MODE == "train":
    training_result = trainer.fit(train_table, snapshot=snapshot)
    disk_checkpoint("model fit")
    retained_bundle = ARTIFACT_ROOT / f"model-{uuid4().hex}"
    shutil.copytree(Path(training_result.bundle_dir), retained_bundle)
    print("Retained local bundle:", retained_bundle)
else:
    print("Reload route: fit skipped.")


### Inspect model metrics and artifacts

In [ ]:
if training_result is None:
    print("Reloaded from:", retained_bundle)
    print("Saved training-table plan:", trainer.training_table_generation_plan)
else:
    print("Available metrics:", training_result.metrics)
    print("Reusable local bundle:", retained_bundle)
    for name, path in training_result.artifact_paths.items():
        print(name, path)
print("Download the retained bundle before Colab recycles the runtime.")


## Generate Batch Predictions

After training a model, generate batch predictions and save their Parquet files under `ARTIFACT_ROOT`. Colab storage is temporary; copy or download the files if they need to persist outside this runtime.

These are predictions for the 90 days after the scoring boundary derived from the latest supplied `OrderDate`, not an accuracy evaluation: the supplied data contains no observed outcomes after that boundary. The query scores customers with an order in the preceding 365 days.

In [ ]:
print('OrderDate range:', *date_bounds)
print('Configured prediction boundary:', PREDICTION_TIME)
print('Population: customers with an order in the preceding 365 days (90-day repeat-order classification)')

### Generate the scoring population

For this temporal query, pass a timezone-naive UTC timestamp and reuse the fitted trainer's saved training-table generation plan.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)
print('Scoring population rows:', prediction_table.row_count)
if not prediction_table.row_count:
    raise RuntimeError(
        'No customers qualify for scoring at the derived boundary. '
        'Check the order dates, customer links, and past-365-day filter.'
    )
disk_checkpoint('prediction-table generation')

### Score locally

In [ ]:
from kumoapi.task import TaskType

prediction_options = {}
if pquery.get_task_type() == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    output_config=None, **prediction_options,
)
print("Rows:", prediction_result.num_rows)
print("Local Parquet files:", prediction_result.files)
disk_checkpoint("prediction")


In [ ]:
if SAVE_PREDICTIONS:
    prediction_dir = ARTIFACT_ROOT / f"predictions-{uuid4().hex}"
    prediction_dir.mkdir(parents=True)
    saved_files = []
    for index, source in enumerate(prediction_result.files):
        target = prediction_dir / f"part-{index}.parquet"
        shutil.copy2(source, target)
        saved_files.append(target)
    manifest = {
        "kumo_engine_version": version("kumo-engine"),
        "prediction_time": PREDICTION_TIME.isoformat(),
        "prediction_rows": prediction_result.num_rows,
        "model_bundle": str(retained_bundle),
        "metrics": training_result.metrics if training_result else None,
        "prediction_files": [str(path) for path in saved_files],
    }
    (prediction_dir / "run.json").write_text(json.dumps(manifest, indent=2, default=str))
    print("Saved prediction directory:", prediction_dir)
    print("Download predictions and the model bundle before the Colab runtime ends.")
else:
    print("SDK prediction Parquet remains in local scratch for this runtime.")


### Read a bounded preview

Read at most ten rows across the local Parquet files in batches. This avoids collecting the entire result.

In [ ]:
import pyarrow.parquet as pq

def preview_parquet(files, limit=10):
    frames = []
    remaining = limit
    for path in files:
        if remaining <= 0:
            break
        with pq.ParquetFile(path) as parquet:
            for batch in parquet.iter_batches(batch_size=remaining):
                frames.append(batch.slice(0, remaining).to_pandas())
                remaining -= len(frames[-1])
                if remaining <= 0:
                    break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

display(preview_parquet(prediction_result.files))